# Lecture 3 follow-along — Python That Behaves Like a Robot

Every code cell is **verbatim** from the slides, in slide order — the same cells you typed into `lec3_warmup.py`, `lec3_sensor.py`, and `lec3_filter.py`. In the container, plots become **files** (`savefig`); here they also render inline.

## Warm-up (Week 0 muscle)
*NEW FILE · COMPLETE — `lec3_warmup.py`: type it, **predict the output**, then run*  
`→ lec3_warmup.py`

In [ ]:
readings = [1.2, 2.6, 0.4, 2.9, 1.1]   # depth samples [m]
count = 0                               # the tally — born at ZERO, nothing counted yet
for r in readings:                      # visit each reading in turn
    if r > 2.0:                         # deeper than 2 m?
        count += 1                      # count it (= count + 1)
print(count, "readings over 2 m")

## Build File 1 — `lec3_sensor.py` (type along)
*NEW FILE · COMPLETE — new file `lec3_sensor.py` in your course folder → type → save (⌘S) → terminal: `python3 lec3_sensor.py`*  
`→ lec3_sensor.py`

In [ ]:
import numpy as np                    # numerical toolbox, nicknamed np
rng = np.random.default_rng(7)        # lie factory — seed 7: same lies every run

t = np.arange(0, 20, 0.1)             # t: 0.0, 0.1, ... 19.9 (20 s at 10 Hz)
true_depth = 2.0 + 0.0*t              # d_true — flat 2 m, one value per tick
z = true_depth + rng.normal(0, 0.08, t.size)   # z_k = d_true + ε_k
print("first five readings:", np.round(z[:5], 3))

## See the lie
*ADD TO `lec3_sensor.py` — run again, then click **lie.png** in the Explorer*  
`→ lec3_sensor.py`

In [ ]:
import matplotlib.pyplot as plt                            # plotting library, nicknamed plt
plt.plot(t, z, '.', ms=3, label='sensor $z_k$')            # '.' dots, markersize 3
plt.plot(t, true_depth, 'k--', label='truth $d_{true}$')   # 'k--' = black, dashed
plt.gca().invert_yaxis()                                   # COURSE RULE: deeper = lower on page
plt.xlabel('time [s]'); plt.ylabel('depth [m] (down = deeper)'); plt.legend(fontsize=8)
plt.savefig("lie.png"); plt.show()   # savefig: the container has no pop-up windows — plots become FILES

## Build File 2 — `lec3_filter.py`
*NEW FILE · COMPLETE — type it, save, run: `python3 lec3_filter.py`*  
`→ lec3_filter.py`

In [ ]:
def moving_average(z, N):
    out = []                    # filtered result, tick by tick
    window = []                 # the last N readings — a moving bucket
    for zi in z:                # for each new reading...
        window.append(zi)       #   drop it in the bucket
        if len(window) > N:     #   bucket over-full?
            window.pop(0)       #   remove the OLDEST (index 0)
        out.append(sum(window) / len(window))   # the mean: the equation, wearing code
    return out                  # one calm number per tick, kept

test = [1, 1, 1, 10, 1, 1]          # one wild spike in calm readings
print(moving_average(test, 3))      # predict first, then run

## Does it work? — bring the data
*ADD TO `lec3_filter.py` — the sensor data, same seed; and the DRY axis helper*  
`→ lec3_filter.py`

In [ ]:
import numpy as np                    # same toolboxes as lec3_sensor.py
import matplotlib.pyplot as plt
rng = np.random.default_rng(7)        # seed 7 again: the SAME lies, replayed
t = np.arange(0, 20, 0.1)             # the same 20 s of ticks
true_depth = 2.0 + 0.0*t              # the same flat 2 m truth
z = true_depth + rng.normal(0, 0.08, t.size)   # the same lying sensor

def depth_axis():                     # the three repeated axis lines, bottled once
    plt.gca().invert_yaxis()          # course rule: deeper = lower on the page
    plt.xlabel('time [s]'); plt.ylabel('depth [m] (down = deeper)')
    plt.legend(fontsize=8)            # one call now dresses every plot

## Does it work?
*ADD TO `lec3_filter.py` — run, click **filter.png***  
`→ lec3_filter.py`

In [ ]:
plt.figure(figsize=(8.2, 2.0))                              # fresh canvas (AUV-012)
plt.plot(t, z, '.', ms=3, alpha=0.4, label='raw')           # raw first — the backdrop           # alpha: 40% opaque
plt.plot(t, moving_average(z, 10), lw=2, label='filtered N=10')  # lw: line width
plt.plot(t, true_depth, 'k--', label='truth')
depth_axis()   # our three repeated axis lines, bottled once
plt.savefig("filter.png"); plt.show()

## No free lunch
*ADD TO `lec3_filter.py` — the dive; run, click `dive.png`*  
`→ lec3_filter.py`

In [ ]:
true2 = np.where(t < 8, 1.0, 2.0)    # np.where(cond, a, b): array-wide if/else
z2 = true2 + rng.normal(0, 0.08, t.size)

plt.figure(figsize=(8.2, 2.0)); plt.plot(t, z2, '.', ms=3, alpha=0.35, label='raw')
plt.plot(t, moving_average(z2, 10), lw=2, label='N=10')
plt.plot(t, moving_average(z2, 40), lw=2, label='N=40')
plt.plot(t, true2, 'k--', label='truth')
plt.annotate('N=40 still climbing —\nreporting the PAST', xy=(10.3, 1.72), xytext=(12.3, 1.15),
    fontsize=8, color='#9a5400', arrowprops=dict(arrowstyle='->', color='#9a5400'))
depth_axis()                         # annotate: arrow FROM the words TO xy
plt.savefig("dive.png"); plt.show()

## One more tool — the depth alarm
*ADD TO `lec3_filter.py` — the second function Lab 2 asks for*  
`→ lec3_filter.py`

In [ ]:
def depth_alarm(t, filtered, limit):
    for i in range(len(filtered)):   # walk the ticks in order
        if filtered[i] > limit:      # deeper than allowed? (down = bigger)
            return t[i]              # return the TIME — and stop: FIRST crossing
    return None                      # loop survived: never crossed

print("N=10 alarm at t =", depth_alarm(t, moving_average(z2, 10), 1.5))
print("N=40 alarm at t =", depth_alarm(t, moving_average(z2, 40), 1.5))